# **Feature Engineering**

## Objectives

* Split the data into train and test sets
* Build a reproducible preprocessing pipeline that implements the cleaning actions and feature transformations explored and validated in the[cleaning](/jupyter_notebooks/04_cleaning.ipynb) and [feature exploration](/jupyter_notebooks/06_feature_exploration.ipynb) notebooks as well as required encoding
* Apply SmartCorrelatedFeatures to identify and reduce redundant correlated fatures
* Fit the pipeline to on train data and transform both train and test sets 


## Inputs

* outputs/datasets/cleaned/HotelBookingsValid.csv
* outputs/correlation/TopFeatures.csv

## Outputs

* Feature engineering pipeline saved as preprocessing_pipeline.pkl
* X_train, X_test, y_train and y_test to outputs/ml_pipeline/cancel_predict/v1 as .csv

## Additional Comments

* ⚠️ TBC ⚠️


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'/home/niall/PP4/cancel-protect/jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

'/home/niall/PP4/cancel-protect'

# Load Data

In [3]:
import pandas as pd

df = pd.read_csv("outputs/datasets/cleaned/HotelBookingsValid.csv")
df.head(3)

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0


In [4]:
top_features = pd.read_csv("outputs/correlation/TopFeatures.csv")
top_features.head(3)

,Unnamed: 0,Feature,Raw/Engineered,Pearson,Spearman,PPS,Above Threshold
0,1,lead_time,Raw,0.292693,0.316357,0.215872,Yes
1,12,country,Raw,NaN,NaN,0.307594,Yes
2,13,market_segment,Raw,NaN,NaN,0.208675,Yes


---

## Split train and test set

In [5]:
from sklearn.model_selection import train_test_split

df_data = df.drop("is_canceled", axis=1)
target = df["is_canceled"]

print(f"df_data shape: {df_data.shape}, target shape: {target.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    df_data, target, test_size=0.2, random_state=4, stratify=target
)

display_shapes = pd.Series({"X_train": X_train.shape,
              "X_test": X_test.shape,
              "y_train": y_train.shape,
              "y_test": y_test.shape}, name="split_shapes")
display_shapes

df_data shape: (119180, 29), target shape: (119180,)


X_train    (95344, 29)
X_test     (23836, 29)
y_train       (95344,)
y_test        (23836,)
Name: split_shapes, dtype: object

---

## Pipeline

**Pipeline actions**

|Feature|Meaning|Data type|Preprocessing actions|Prediction model actions|Experimental prediction model alternatives|Cluster model actions|Experimental cluster model alternatives|
|---|---|---|---|---|---|---|---|
|hotel|booking location|nominal|one-hot|||||
|is_canceled|booking was cancelled|binary|none|target only|SMOTE|exclude||
|lead_time|days before arrival booking made|numeric|none||log/skew transformation; binning|scaling|log/skew transformation|
|arrival_date_year|year of arrival|numeric|drop|exclude||exclude||
|arrival_date_month|month of arrival|nominal|defer|one-hot|cyclical encoding|cyclical encoding||
|arrival_date_week_number|week of the year of arrival|numeric|none||cyclical encoding|||
|arrival_date_day_of_month|day in the month of arrival|numeric|none||||exclude|
|stays_in_weekend_nights|weekend nights stayed|numeric|none||binning|scaling|binning|
|stays_in_week_nights|week nights stayed|numeric|none||binning|scaling|binning|
|adults|number of adults on the booking|numeric|none||binning|scaling|binning|
|children|number of children|numeric|none||binary, binning|binning||
|babies|number of babies|numeric|none||binary, binning|binning||
|meal|meal plan booked|nominal|replace "Undefined" with "SC", one-hot|||||
|country|country of origin|nominal|impute missing data|ordinal-categorical|frequency, target|frequency|regional grouping, exclude|
|market_segment|demographic information|nominal|one-hot|||||
|distribution_channel|demographic information|nominal|one-hot|||||
|is_repeated_guest|if the guest has booked before|binary|none|||||
|previous_cancellations|how many times the guest has cancelled before|numeric|none||binary|scaling|binary|
|previous_bookings_not_canceled|how many times the guest has completed a booking|numeric|none||log/skew transformation; binning|scaling|log/skew transformation; binning|
|reserved_room_type|Room code booked|nominal|one-hot|||||
|assigned_room_type|Room code assigned|nominal|one-hot|||||
|booking_changes|How many alterations were made to the booking|numeric|none||binary|scaling||
|deposit_type|Booking security policy|nominal|one-hot|||||
|agent|ID code of booking agent|nominal|impute missing data||frequency, target, exclude|exclude||
|company|ID code of company the guest is travelling for|nominal|drop|||||
|days_in_waiting_list|How long the booking waited for confirmation|numeric|none||binary, skew transformation|scaling|binary|
|customer_type|demographic information|nominal|one-hot|||||
|adr|cost per night of the booking|numeric|none||log/skew transformation; binning|scaling|log/skew transformation; binning|
|required_car_parking_spaces|car parking spaces needed|numeric|none||binary|scaling||
|total_of_special_requests|special requests made|numeric|none||binary|scaling||

**Cleaning steps**
1. Drop `company` and `arrival_date_year`
2. Replace "Undefined" wit "SC" in `meal`

* Create preprocessing pipeline

In [9]:
print("X_train shape ", X_train.shape)

X_train shape  (95344, 29)


In [32]:
from feature_engine.selection import DropFeatures

X_train_copy = X_train.copy()
pipeline_step1 = DropFeatures(features_to_drop=["company", "arrival_date_year"])
pipeline_step1 = pipeline_step1.fit_transform(X_train_copy)
pipeline_step1.shape


(95344, 27)

In [12]:
def undefined_meal(data):
    data = data.copy()
    data["meal"] = data["meal"].replace("Undefined", "SC")
    return data

In [ ]:
from sklearn.preprocessing import FunctionTransformer

pipeline_step2 = FunctionTransformer(undefined_meal)
pipeline_step2 = pipeline_step2.fit_transform(pipeline_step1)
pipeline_step2["meal"].value_counts()

meal
BB    73807
HB    11541
SC     9357
FB      639
Name: count, dtype: int64

In [34]:
pipeline_step2["agent"].describe()

count    82223.000000
mean        86.575824
std        110.663139
min          1.000000
25%          9.000000
50%         14.000000
75%        229.000000
max        535.000000
Name: agent, dtype: float64

In [35]:
pipeline_step2["agent"].isnull().sum()

np.int64(13121)

In [36]:
from feature_engine.imputation import ArbitraryNumberImputer, CategoricalImputer

pipeline_step3 = ArbitraryNumberImputer(arbitrary_number=0, variables="agent")
pipeline_step3 = pipeline_step3.fit_transform(pipeline_step2)
print("Missing values: ", pipeline_step3["agent"].isnull().sum())
pipeline_step3["agent"].describe()


Missing values:  0


count    95344.000000
mean        74.661478
std        107.007270
min          0.000000
25%          7.000000
50%          9.000000
75%        152.000000
max        535.000000
Name: agent, dtype: float64

In [37]:
categorical_cols = ["hotel", "meal", "market_segment", "distribution_channel", "reserved_room_type", "assigned_room_type", "deposit_type", "customer_type"]
for col in categorical_cols:
    print(f"{col} has {df[col].nunique()} categories")

hotel has 2 categories
meal has 5 categories
market_segment has 7 categories
distribution_channel has 4 categories
reserved_room_type has 9 categories
assigned_room_type has 11 categories
deposit_type has 3 categories
customer_type has 4 categories


In [39]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
    ],
    remainder="passthrough"
)
pipeline_step4 = preprocessor.fit_transform(pipeline_step3)
pipeline_step4.shape

(95344, 63)

In [ ]:
from sklearn.pipeline import Pipeline

def preprocessing_pipeline():

    def undefined_meal(data):
        data = data.copy()
        data["meal"] = data["meal"].replace("Undefined", "SC")
        return data

    pipeline_base = Pipeline([
        ("DropFeatures", DropFeatures(features_to_drop=["company", "arrival_date_year"])),
        ("FunctionTransformer", FunctionTransformer(undefined_meal)),
        ()
    ])

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In case you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create here your folder
  # os.makedirs(name='')
except Exception as e:
  print(e)
